In [2]:
import os
import random

import numpy as np
import torch
from datasets import load_dataset
from peft import LoraConfig, TaskType, get_peft_model
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
)
from torch.nn.utils.rnn import pad_sequence

In [3]:
# ============================================================
# Configuration
# ============================================================

MODEL_NAME = "openai-community/gpt2"
DATASET_NAME = "nshah-fbcs/childes-engUK-conversational-pairs"

MAX_TRAIN_EXAMPLES = 10_000
MAX_VAL_EXAMPLES = 1_000
MAX_TEST_EXAMPLES = 1_000

MAX_LENGTH = 1024

BATCH_SIZE = 4
NUM_EPOCHS = 1

LEARNING_RATE = 5e-4
WEIGHT_DECAY = 0.01

SEED = 42

OUTPUT_DIR = "./gpt2-childes-lora-c1"

In [4]:
# ============================================================
# Reproducibility
# ============================================================

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


In [5]:
# ============================================================
# Device
# ============================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [6]:
# ============================================================
# Tokenizer
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# GPT-2 does not have a separate PAD token.
# We use EOS as PAD, as in Experiment B.
tokenizer.pad_token = tokenizer.eos_token


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [7]:
# ============================================================
# Dataset
# ============================================================

dataset = load_dataset(DATASET_NAME)

train_dataset = dataset["train"].select(
    range(min(MAX_TRAIN_EXAMPLES, len(dataset["train"])))
)

val_dataset = dataset["validation"].select(
    range(min(MAX_VAL_EXAMPLES, len(dataset["validation"])))
)

test_dataset = dataset["test"].select(
    range(min(MAX_TEST_EXAMPLES, len(dataset["test"])))
)


README.md:   0%|          | 0.00/5.01k [00:00<?, ?B/s]

train.jsonl: reconstructing file:   0%|          |  0.00B / 34.1MB            

train.jsonl: downloading bytes:           |  0.00B            

validation.jsonl:   0%|          | 0.00/1.79M [00:00<?, ?B/s]

test.jsonl:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/250757 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/13197 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/14504 [00:00<?, ? examples/s]

In [8]:
# ============================================================
# Convert each example to:
#
# Adult: ...
# Child: ...
#
# with response-only labels.
# ============================================================

def get_adult_child(example):
    if example["speaker_a"] == "adult":
        adult_utterance = example["utterance_a"]
        child_utterance = example["utterance_b"]

    elif example["speaker_b"] == "adult":
        adult_utterance = example["utterance_b"]
        child_utterance = example["utterance_a"]

    else:
        raise ValueError(
            f"Could not find adult speaker in example: {example}"
        )

    return adult_utterance, child_utterance


In [9]:
def tokenize_example(example):
    adult_utterance, child_utterance = get_adult_child(example)

    prompt = f"Adult: {adult_utterance}\nChild:"
    response = f" {child_utterance}{tokenizer.eos_token}"

    prompt_ids = tokenizer(
        prompt,
        add_special_tokens=False,
    )["input_ids"]

    response_ids = tokenizer(
        response,
        add_special_tokens=False,
    )["input_ids"]

    input_ids = prompt_ids + response_ids

    # Response-only SFT:
    #
    # prompt tokens -> ignored by cross entropy
    # response tokens -> contribute to loss
    labels = [-100] * len(prompt_ids) + response_ids


    # Truncate both together.
    input_ids = input_ids[:MAX_LENGTH]
    labels = labels[:MAX_LENGTH]

    return {
        "input_ids": input_ids,
        "labels": labels,
    }

In [10]:
train_dataset = train_dataset.map(
    tokenize_example,
    remove_columns=train_dataset.column_names,
)

val_dataset = val_dataset.map(
    tokenize_example,
    remove_columns=val_dataset.column_names,
)

test_dataset = test_dataset.map(
    tokenize_example,
    remove_columns=test_dataset.column_names,
)

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [11]:
# ============================================================
# Custom collator
#
# We keep this from Experiment B because the response-only
# label construction is an important part of our experiment.
# ============================================================

def collate_fn(batch):
    input_ids = [
        torch.tensor(x["input_ids"], dtype=torch.long)
        for x in batch
    ]

    labels = [
        torch.tensor(x["labels"], dtype=torch.long)
        for x in batch
    ]

    input_ids = pad_sequence(
        input_ids,
        batch_first=True,
        padding_value=tokenizer.pad_token_id,
    )


    labels = pad_sequence(
        labels,
        batch_first=True,
        padding_value=-100,
    )

    attention_mask = (
        input_ids != tokenizer.pad_token_id
    ).long()

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
    }


In [15]:
!pip install -U torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 36.4 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [16]:
# ============================================================
# Load GPT-2
# ============================================================

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

# Make sure GPT-2 knows its padding token.
model.config.pad_token_id = tokenizer.pad_token_id


# ============================================================
# C1: LoRA only on the MLP of the final Transformer block
#
# GPT-2 block 11:
#
# transformer.h.11.mlp.c_fc
# transformer.h.11.mlp.c_proj
#
# PEFT's target_modules matching is name-based. Because
# c_fc/c_proj occur in every block, we first verify the names
# and then use exact module names for block 11.
# ============================================================

target_modules = [
    "transformer.h.11.mlp.c_fc",
    "transformer.h.11.mlp.c_proj",
]

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=target_modules,
    bias="none",
)

model = get_peft_model(model, lora_config)

print("\nTrainable parameters:")
model.print_trainable_parameters()


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]


Trainable parameters:
trainable params: 61,440 || all params: 124,501,248 || trainable%: 0.0493


/usr/local/lib/python3.13/dist-packages/peft/tuners/lora/layer.py:2631: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


In [17]:
# ============================================================
# Verify exactly which parameters are trainable
# ============================================================

print("\nTrainable parameter names:")

for name, param in model.named_parameters():
    if param.requires_grad:
        print(f"  {name}  {tuple(param.shape)}")


Trainable parameter names:
  base_model.model.transformer.h.11.mlp.c_fc.lora_A.default.weight  (8, 768)
  base_model.model.transformer.h.11.mlp.c_fc.lora_B.default.weight  (3072, 8)
  base_model.model.transformer.h.11.mlp.c_proj.lora_A.default.weight  (8, 3072)
  base_model.model.transformer.h.11.mlp.c_proj.lora_B.default.weight  (768, 8)


In [18]:
# ============================================================
# Inspect one batch
# ============================================================

print("\nInspecting one training batch...")

sample_batch = collate_fn(
    [train_dataset[i] for i in range(min(BATCH_SIZE, len(train_dataset)))]
)

print("input_ids shape:     ", sample_batch["input_ids"].shape)
print("attention_mask shape:", sample_batch["attention_mask"].shape)
print("labels shape:        ", sample_batch["labels"].shape)

print("\nFirst input:")
print(
    tokenizer.decode(
        sample_batch["input_ids"][0],
        skip_special_tokens=False,
    )
)

response_label_ids = sample_batch["labels"][0]
response_label_ids = response_label_ids[
    response_label_ids != -100
]

print("\nFirst response labels:")
print(
    tokenizer.decode(
        response_label_ids,
        skip_special_tokens=False,
    )
)


Inspecting one training batch...
input_ids shape:      torch.Size([4, 27])
attention_mask shape: torch.Size([4, 27])
labels shape:         torch.Size([4, 27])

First input:
Adult: i don't know .
Child: &~w what's that ?<|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|>

First response labels:
 &~w what's that ?<|endoftext|>


In [21]:
# ============================================================
# Trainer
# ============================================================

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,

    num_train_epochs=NUM_EPOCHS,

    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,

    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,

    eval_strategy="epoch",
    save_strategy="epoch",

    logging_steps=100,

    report_to="none",

    seed=SEED,

    # Keep the experiment simple on the Mac.
    fp16=False,
    bf16=False,
)


trainer = Trainer(
    model=model,
    args=training_args,

    train_dataset=train_dataset,
    eval_dataset=val_dataset,

    data_collator=collate_fn,
)



In [22]:
# ============================================================
# Train
# ============================================================

print("\nStarting C1 LoRA training...")

trainer.train()




Starting C1 LoRA training...


[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss
1,3.281524,3.163880


TrainOutput(global_step=2500, training_loss=3.327322229003906, metrics={'train_runtime': 61.1145, 'train_samples_per_second': 163.627, 'train_steps_per_second': 40.907, 'total_flos': 130325695672320.0, 'train_loss': 3.327322229003906, 'epoch': 1.0})

In [23]:
# ============================================================
# Evaluation
# ============================================================

print("\nEvaluating...")

eval_results = trainer.evaluate()

print("\nEvaluation results:")
for key, value in eval_results.items():
    print(f"{key}: {value}")


if "eval_loss" in eval_results:
    perplexity = np.exp(eval_results["eval_loss"])
    print(f"eval_perplexity: {perplexity:.4f}")




Evaluating...


Training Loss,Validation Loss,Epoch
3.281524,3.163880,1



Evaluation results:
eval_loss: 3.1638801097869873
eval_perplexity: 23.6622


In [24]:
# ============================================================
# Save LoRA adapter
# ============================================================

print("\nSaving LoRA adapter...")

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)




Saving LoRA adapter...


('./gpt2-childes-lora-c1/tokenizer_config.json',
 './gpt2-childes-lora-c1/tokenizer.json')

In [25]:
# ============================================================
# Generation
# ============================================================

def generate_child_response(
    model,
    adult_utterance,
    max_new_tokens=50,
):
    model.eval()

    prompt = f"Adult: {adult_utterance}\nChild:"

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
    )

    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    prompt_length = inputs["input_ids"].shape[1]

    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,

            max_new_tokens=max_new_tokens,

            do_sample=True,
            temperature=0.8,
            top_k=50,

            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    # Remove the prompt from the generated sequence.
    new_token_ids = generated_ids[0, prompt_length:]

    response = tokenizer.decode(
        new_token_ids,
        skip_special_tokens=False,
    )


    return response



In [26]:
# ============================================================
# Sample generations
# ============================================================

test_prompts = [
    "what did you do today ?",
    "what are you playing with ?",
    "where is mommy ?",
    "what did you eat ?",
    "do you want to go outside ?",
]

print("\nSample generations:\n")

for prompt in test_prompts:
    response = generate_child_response(model, prompt)

    print(f"Adult: {prompt}")
    print(f"Child:{response}")
    print("-" * 60)




Sample generations:

Adult: what did you do today ?
Child: like a baby .<|endoftext|>
------------------------------------------------------------
Adult: what are you playing with ?
Child: no it's just going to be a normal two man game .<|endoftext|>
------------------------------------------------------------
Adult: where is mommy ?
Child: where mommy .<|endoftext|>
------------------------------------------------------------
Adult: what did you eat ?
Child: what did you eat ?<|endoftext|>
------------------------------------------------------------
Adult: do you want to go outside ?
Child: you want to go outside ?<|endoftext|>
------------------------------------------------------------


In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
!jupyter nbconvert --to html /content/drive/MyDrive/"Colab Notebooks"/sft_childes_gpt2_lora_c1.ipynb --output-dir=/content/drive/MyDrive/


[NbConvertApp] Converting notebook /content/drive/MyDrive/Colab Notebooks/sft_childes_gpt2_lora_c1.ipynb to html
[NbConvertApp] Writing 491897 bytes to /content/drive/MyDrive/sft_childes_gpt2_lora_c1.html


In [6]:
!pwd

/content


In [7]:
!ls -l sample_data

total 55504
-rwxr-xr-x 1 root root     1697 Jan  1  2000 anscombe.json
-rw-r--r-- 1 root root   301141 Sep  4 13:32 california_housing_test.csv
-rw-r--r-- 1 root root  1706430 Sep  4 13:32 california_housing_train.csv
-rw-r--r-- 1 root root 18289443 Sep  4 13:32 mnist_test.csv
-rw-r--r-- 1 root root 36523880 Sep  4 13:32 mnist_train_small.csv
-rwxr-xr-x 1 root root      962 Jan  1  2000 README.md
